# 02 · Wilson Confidence Intervals and Standard Error

# Project: Integrating Moral Values in Turkish EFL Classrooms

These notebooks are a reproducible analysis companion for the supplied project materials. They do not invent participant-level records. Appendix E/F provide aggregate frequencies with N=20; separate questionnaire notes contain percentages without an explicit denominator and conflict with the appendices on some items. Treat each source stream separately. The main manuscript describes a larger mixed-methods sample (the supplied abstract is truncated), so these appendix counts must not be silently generalized to the manuscript sample.

Run from any working directory. Generated files go to `./analysis_outputs` relative to the current working directory. Python 3.9+; dependencies: pandas, numpy, matplotlib (Notebook 3 only).

## Estimands and scope
Calculate Wilson score 95% confidence intervals and binomial standard errors for appendix proportions. These are illustrative uncertainty summaries conditional on the appendix N=20 counts and a binomial sampling model. They do not resolve the sample-size conflict, non-probability sampling, dependence, or measurement limitations.

In [ ]:
import math
import pandas as pd
from statistics import NormalDist

## Methods
For observed proportion p=x/n and z=1.95996, Wilson limits are computed by the score-interval formula. The standard error is sqrt[p(1-p)/n]. For an exhaustive multi-category question these are category-wise marginal intervals; they are not simultaneous intervals and categories are dependent.

In [ ]:
Z = NormalDist().inv_cdf(0.975)
def wilson(x, n, z=Z):
    if not (0 <= x <= n and n > 0): raise ValueError('Require 0 <= x <= n and n > 0')
    p=x/n; den=1+z*z/n
    center=(p+z*z/(2*n))/den
    half=z*math.sqrt(p*(1-p)/n+z*z/(4*n*n))/den
    return center-half, center+half

def summarize(x,n):
    p=x/n; lo,hi=wilson(x,n)
    return {'count':x,'n':n,'proportion':p,'percent':100*p,'se':math.sqrt(p*(1-p)/n),'wilson95_low':lo,'wilson95_high':hi}

## Appendix E teacher indicators
Intervals are calculated from the supplied category counts. Category percentages within each item sum to 100%, but their intervals should not be summed.

In [ ]:
items = [
 ('Q1 fully/considerably responsible',14,20),('Q2 Global citizenship',14,20),
 ('Q3 insufficient curriculum guidance',16,20),('Q5 debates',9,20),
 ('Q7 student resistance',10,20),('Q7 time constraints',6,20),('Q7 lack of training',4,20),
]
results=pd.DataFrame([{'indicator':name,**summarize(x,n)} for name,x,n in items])
results[['percent','se','wilson95_low','wilson95_high']] = results[['percent','se','wilson95_low','wilson95_high']].round(4)
results['wilson95_low_pct']=100*results.wilson95_low
results['wilson95_high_pct']=100*results.wilson95_high
print(results.to_string(index=False))

## Sensitivity and checks
At small n, normal/Wald intervals can perform poorly, especially near 0 or 1; Wilson is preferred here. SE remains the plug-in binomial SE. No interval is calculated for the unsourced percentages without a denominator. The reference calculation below checks boundary behavior.

In [ ]:
assert wilson(0,20)[0] >= 0 and wilson(0,20)[1] > 0
assert wilson(20,20)[1] <= 1 and wilson(20,20)[0] < 1
assert abs(summarize(14,20)['proportion']-.70)<1e-12
assert abs(summarize(14,20)['se']-math.sqrt(.7*.3/20))<1e-12
print('Checks passed; 14/20 = 70%, SE =',round(summarize(14,20)['se'],4),'95% Wilson =',tuple(round(v,4) for v in wilson(14,20)))

## Reporting note
Describe these as approximate binomial intervals for appendix summary proportions—not as definitive population estimates. Confirm the actual denominator and sampling design from primary data before manuscript use.